# 04 - Parametros de generacion del modelo final

El notebook 03 eligio **como se entrena** el modelo. Este elige **como se
muestrea de el**, que es una decision distinta y posterior: el adaptador ya esta
congelado y no se toca.

Los siete parametros que intervienen, y que hace cada uno:

| Parametro | Que controla | Decision |
|---|---|---|
| `do_sample` | si se muestrea o se toma siempre el token mas probable | **se barre** |
| `temperature` | cuanto se aplana la distribucion antes de muestrear | **se barre** |
| `top_p` | se muestrea solo del nucleo que acumula esa probabilidad | fijo en 0.9 |
| `top_k` | se muestrea solo de los k tokens mas probables | fijo en 50 |
| `repetition_penalty` | castiga repetir tokens ya emitidos | **se barre** |
| `max_new_tokens` | cuantos tokens NUEVOS puede escribir | fijo en 300 |
| `max_tokens` | *no existe en `transformers`* | ver nota |

**Sobre `top_p` y `top_k`.** Se dejan en 0.9 y 50, los valores de referencia.
No es pereza: los tres recortan la misma cola de la distribucion, asi que
barrerlos junto con la temperatura mide el mismo efecto tres veces. La
temperatura es la que de verdad mueve la aguja, y con ella se barre.

**Sobre `max_tokens`.** Es el nombre que usa la API de OpenAI. En `transformers`
el equivalente es `max_new_tokens` (tokens generados) y existe ademas
`max_length` (prompt + generacion). Se usa `max_new_tokens` porque no depende
del largo del fragmento, que aqui varia mucho.

**Sobre `max_new_tokens=300`.** Una pregunta con cuatro opciones en JSON ronda
los 150-200 tokens. 300 deja margen sin permitir que el modelo divague. El
barrido cuenta cuantas generaciones se **truncan** al llegar al tope: si fueran
muchas, el valor estaria mal elegido.

In [1]:
import json
import re
import time
import unicodedata
from pathlib import Path

import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO = "Qwen/Qwen3-4B-Instruct-2507"
ADAPTADOR = Path("../app/adapter")
DATA = Path("../data")
N_CASOS = 20
REPETICIONES = 3        # para medir variedad hay que regenerar el mismo fragmento

INSTRUCCION = (
    "Eres un docente de medicina. A partir del FRAGMENTO escribe UNA pregunta "
    "de opcion multiple en espanol neutro. Responde SOLO con JSON."
)

# top_p=0.9 y top_k=50 son los valores estandar y quedan fijos como referencia.
# Se varia lo que de verdad mueve la aguja: la temperatura y la penalizacion.
CONFIGS = [
    dict(nombre="greedy (referencia)", do_sample=False),
    dict(nombre="temp 0.3", do_sample=True, temperature=0.3, top_p=0.9, top_k=50),
    dict(nombre="temp 0.7", do_sample=True, temperature=0.7, top_p=0.9, top_k=50),
    dict(nombre="temp 1.0", do_sample=True, temperature=1.0, top_p=0.9, top_k=50),
    dict(nombre="temp 0.7 + rep 1.1", do_sample=True, temperature=0.7, top_p=0.9,
         top_k=50, repetition_penalty=1.1),
    dict(nombre="temp 0.7 + rep 1.2", do_sample=True, temperature=0.7, top_p=0.9,
         top_k=50, repetition_penalty=1.2),
    dict(nombre="temp 0.7 sin top_k", do_sample=True, temperature=0.7, top_p=0.9),
]

MAX_NEW = 300


W0909 19:25:54.338000 2380 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [2]:
def sin_acentos(s):
    return "".join(c for c in unicodedata.normalize("NFD", str(s).lower())
                   if unicodedata.category(c) != "Mn")


def parsear(t):
    t = t.strip()
    if t.startswith("```"):
        t = t.strip("`").removeprefix("json").strip()
    try:
        return json.loads(t)
    except Exception:
        return None


## La tension que hay que resolver

No hay una configuracion buena en abstracto; hay dos objetivos enfrentados:

- **Formato.** El modelo debe devolver JSON parseable con cuatro opciones.
  `do_sample=False` (greedy) es lo mas seguro para esto.
- **Variedad.** El boton *"otro examen del mismo tema"* no sirve de nada si el
  mismo fragmento devuelve siempre la misma pregunta. Greedy es determinista:
  variedad exactamente 1 de 3.

Por eso cada fragmento se genera **tres veces** y se cuentan cuantas preguntas
**distintas** salieron. La configuracion elegida es la de mayor variedad que no
pierda formato.

In [3]:
tok = AutoTokenizer.from_pretrained(ADAPTADOR)
base = AutoModelForCausalLM.from_pretrained(MODELO, dtype=torch.bfloat16, device_map="cuda")
model = PeftModel.from_pretrained(base, ADAPTADOR)
model.eval()

test = pd.read_parquet(DATA / "mcq_test.parquet").sample(N_CASOS, random_state=42)
print(f"{N_CASOS} fragmentos x {REPETICIONES} repeticiones x {len(CONFIGS)} configuraciones")
print(f"= {N_CASOS * REPETICIONES * len(CONFIGS)} generaciones\n")


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

20 fragmentos x 3 repeticiones x 7 configuraciones
= 420 generaciones



In [4]:
filas = []
for cfg in CONFIGS:
    nombre = cfg.pop("nombre")
    t0 = time.time()
    validos = completas = truncadas = 0
    variedad = []

    for _, fila in test.iterrows():
        msgs = [{"role": "system", "content": INSTRUCCION},
                {"role": "user", "content": "FRAGMENTO:\n" + fila["chunk_text"]}]
        texto = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ids = tok([texto], return_tensors="pt").to(model.device)

        preguntas = set()
        for _ in range(REPETICIONES):
            with torch.no_grad():
                out = model.generate(**ids, max_new_tokens=MAX_NEW,
                                     pad_token_id=tok.eos_token_id, **cfg)
            nuevos = out.shape[1] - ids.input_ids.shape[1]
            if nuevos >= MAX_NEW:
                truncadas += 1
            d = parsear(tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True))
            if d is None:
                continue
            validos += 1
            if (d.get("pregunta") and d.get("correcta")
                    and isinstance(d.get("incorrectas"), list) and len(d["incorrectas"]) == 3):
                completas += 1
                preguntas.add(sin_acentos(d["pregunta"]).strip())

        # cuantas preguntas DISTINTAS salieron de este mismo fragmento
        variedad.append(len(preguntas))

    total = N_CASOS * REPETICIONES
    filas.append({
        "configuracion": nombre,
        "JSON valido": f"{validos}/{total}",
        "estructura": f"{completas}/{total}",
        "% formato ok": round(completas / total * 100, 1),
        "variedad": round(sum(variedad) / len(variedad), 2),   # de 3 posibles
        "truncadas": truncadas,
        "segundos": round((time.time() - t0) / total, 2),
    })
    print(f"  {nombre:<22} formato {completas}/{total}  "
          f"variedad {sum(variedad)/len(variedad):.2f}/3  "
          f"({(time.time()-t0)/60:.1f} min)", flush=True)


  greedy (referencia)    formato 60/60  variedad 1.00/3  (6.4 min)


  temp 0.3               formato 60/60  variedad 1.95/3  (6.5 min)


  temp 0.7               formato 60/60  variedad 2.30/3  (6.8 min)


  temp 1.0               formato 60/60  variedad 2.70/3  (7.0 min)


  temp 0.7 + rep 1.1     formato 60/60  variedad 2.45/3  (6.9 min)


  temp 0.7 + rep 1.2     formato 60/60  variedad 2.60/3  (6.9 min)


  temp 0.7 sin top_k     formato 60/60  variedad 2.50/3  (7.0 min)


## Como leer la tabla

`variedad` es el promedio de preguntas distintas obtenidas al generar 3 veces el
mismo fragmento: 1.00 significa que siempre sale la misma, 3.00 que las tres
fueron distintas.

`truncadas` cuenta las generaciones que llegaron a los 300 tokens sin cerrar el
JSON. Son perdida pura: producen texto incompleto que no parsea.

In [5]:
tabla = pd.DataFrame(filas)
print("\n" + "=" * 88)
print(tabla.to_string(index=False))
tabla.to_csv(DATA / "barrido_generacion.csv", index=False)

print("\nvariedad = preguntas distintas de 3 generaciones del MISMO fragmento")
print("  1.00 = siempre la misma (sin variedad)")
print("  3.00 = las tres distintas")
print("\nLa mejor configuracion es la de mayor variedad SIN perder formato.")



      configuracion JSON valido estructura  % formato ok  variedad  truncadas  segundos
greedy (referencia)       60/60      60/60         100.0      1.00          0      6.41
           temp 0.3       60/60      60/60         100.0      1.95          0      6.55
           temp 0.7       60/60      60/60         100.0      2.30          0      6.77
           temp 1.0       60/60      60/60         100.0      2.70          0      7.01
 temp 0.7 + rep 1.1       60/60      60/60         100.0      2.45          0      6.94
 temp 0.7 + rep 1.2       60/60      60/60         100.0      2.60          0      6.86
 temp 0.7 sin top_k       60/60      60/60         100.0      2.50          0      7.05

variedad = preguntas distintas de 3 generaciones del MISMO fragmento
  1.00 = siempre la misma (sin variedad)
  3.00 = las tres distintas

La mejor configuracion es la de mayor variedad SIN perder formato.
